In [ ]:
#https://portail-api.meteofrance.fr/web/fr/faq

### Objectifs de cette partie

Le code ci-dessous : 
- Communique avec l'API AROME et récupère les cartes climatiques une par une sur 24h00.
- Convertit chaque carte en un dataframe Spark. 
- Effectue un calcul de distance entre les coordonnées des stations climatiques et les coordonnées de la carte à l'aide de la distance de Haversine dans le but de récupérer les variables climatiques prédites.
- Stocke les informations dans la bdd Cassandra.

Pistes d'améliorations futures avec les ressources adéquates (notamment par rapport à la distribution):

- Concaténer les cartes dans un dataframe global avec un id spécifique pour chaque carte qui permettrait de partitionner les cartes en fonction de l'id.
- Effectuer le calcul de distance entre chacunes des stations et chacunes des cartes de manière distribuée.

Autre piste : 

Ne sélectionner que les stations climatiques les plus pertinentes par rapport à la variable consonmation. Il se peut que les stations à proximité des zones rurales n'aient pas le même impact que celles proches des zones urbaines. 

#### Remarques 

Plusieurs contraintes par rapport à AROME et son API : 
- AROME est une API asynchrone qui ne permet pas de faire plus de 50 appels à la minute.
- AROME génère des cartes au format grib2 qui ne sont pas exploitables en l'état par Spark
- AROME ne permet pas de récupérer un ensemble de carte, il est obligatoire de récupérer les cartes une par une (une carte équivaut à une heure).

### 1- Création de la session Spark 

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf 
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, FloatType, time
from pyspark.sql.functions import year, month, dayofweek, hour, udf, lit, col


spark = SparkSession.builder \
    .appName("Variable_AROME_24h") \
    .config("spark.cassandra.connection.host", "172.17.0.3") \
    .config("spark.cassandra.connection.port", "9042") \
    .config("spark.sql.extensions","com.datastax.spark.connector.CassandraSparkExtensions") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

### - Import des bibliothèques

In [2]:
import json
import requests
import time
import datetime
import os
import eccodes as ecc
from eccodes import *
import time
import datetime
import io

### 2- Listes des fonctions 

#### a- Fonction de Haversine acos

In [3]:
import pyspark.sql.functions as F

def distance(lat_capteur, lon_capteur, lat_arome, lon_arome):
    return F.acos(
        F.sin(F.radians(lat_capteur)) * F.sin(F.radians(lat_arome)) + 
        F.cos(F.radians(lat_capteur)) * F.cos(F.radians(lat_arome)) * 
            F.cos(F.radians(lon_capteur) - F.radians(lon_arome))
    ) * F.lit(6371.0)

#### b- Fonction de Haversine arcsin

import pyspark.sql.functions as F
def distance(lat_capteur, lon_capteur, lat_arome, lon_arome):
    return F.asin(
        F.sqrt(F.pow((F.sin((F.radians(lat_arome)) - F.radians(lat_capteur))/2),2) +
        F.cos(F.radians(lat_capteur))*F.cos(F.radians(lat_arome))*
        F.pow(F.sin(((F.radians(lon_arome)) - F.radians(lon_capteur))/2),2))
    ) * (2*F.lit(6371.0))

#### c- Fonction pour récupérer la saison 

In [4]:
def return_saison(data):
    #Hiver
    if data in [12,1,2]:
        return 4
    #Printemps
    if data in [3,4,5]:
        return 3
    #Ete
    if data in [6,7,8]:
        return 2
    #Automne
    if data in [9,10,11]:
        return 1
# Conversion en UDF pour le calcul parallèle : 
UDFreturn_saison = udf(lambda data: return_saison(data), IntegerType())

#### d- Conversion du fichier grib2

Le code ci-dessous ouvre le fichier grib2 et le convertit le dataframe en Dataframe Spark.

In [5]:
def conversion_grib2(carte,timestamp,liste_stations):
    print(timestamp)
    # Charger les données GRIB en mémoire
    gid = ecc.codes_new_from_message(carte.content)
    latitudes = ecc.codes_get_array(gid, "latitudes")
    longitudes = ecc.codes_get_array(gid, "longitudes")
    values = ecc.codes_get_array(gid, "values")
    data=[]
    # On stocke les données dans un array mais préalablement on élague les lignes avec des valeurs étranges: 
    for lat, lon, val in zip(latitudes, longitudes, values):
        val=round(val- 273.15,1) #Conversion en celcius
        if (val<100 and val is not None):
            data.append((float(lat), float(lon), float(val))) 
    ecc.codes_release(gid)
    # Création du Schema du dataframe pour la conversion de l'array :
    schema = StructType([
        StructField("latitude", FloatType(), True),
        StructField("longitude", FloatType(), True),
        StructField("t", FloatType(), True)
    ])
    # Création du dataframe Spark avec l'array précédemment créé
    dataframe_carte_arome = spark.createDataFrame(data, schema=schema)
    #Itération sur les stations : 
    dataframe_carte_arome.cache()
    for station in liste_stations.take(liste_stations.count()):
        #On applique la fonction qui permet de calculer la distance entre la station concernée et la carte climatique et cela de manière distribuée (si la table est bien partitionnée dans le cluster)
        dfa = dataframe_carte_arome.withColumn(station[0], distance(F.lit(station['latitude']), F.lit(station['longitude']), F.col("Latitude"), F.col("Longitude")))
        #Afin de récupérer la distance minimum :
        valeur_min = dfa.select(F.min(station[0])).collect()[0][0]
        filtre = dfa.filter(dfa[station[0]]==valeur_min)
        #On ajoute les colonnes supplémentaire avant l'écriture dans la base cassandra
        filtre = filtre.withColumn("timestamp",lit(timestamp)).withColumn('id_station',lit(station[0]))
        filtre = filtre.withColumn('annee', year(filtre["timestamp"])).withColumn('mois', month(filtre["timestamp"]))\
        .withColumn('jour', dayofweek(filtre["timestamp"])).withColumn('heure', hour(filtre["timestamp"]))
        filtre = filtre.withColumn('saison', UDFreturn_saison(filtre["mois"]))
        filtre = filtre.select('id_station','timestamp','saison','annee','mois','jour','heure','t')
        filtre.show()
        #Pour finir, on écrit dans la bdd
        try : 
            filtre.write.format("org.apache.spark.sql.cassandra").options(table="mesures_stations_arome", keyspace="consom") \
            .mode("append") \
            .save()
        except Exception as e:
            erreur = f"Erreur lors de l'écriture dans Cassandra : {str(e)}"
            print(erreur)
    dataframe_carte_arome.unpersist()

### 3- Connexion à l'API et récupération des cartes

A noter que le code ci-dessous n'est pour l'instant pas optimisé pour la distribution des opérations. Certains commentaires à ce sujet sont disponibles dans le rapport.

In [ ]:
#Il est nécessaire de créer un compte sur le portail api de meteofrance afin de générer un id : https://portail-api.meteofrance.fr/
APPLICATION_ID = 'mon_id'
# url to obtain acces token
TOKEN_URL = "https://portail-api.meteofrance.fr/token"

class Client(object):

    def __init__(self):
        self.session = requests.Session()

    def request(self, method, url, **kwargs):
        # First request will always need to obtain a token first
        if 'Authorization' not in self.session.headers:
            self.obtain_token()
        # Optimistically attempt to dispatch reqest
        response = self.session.request(method, url, **kwargs)

        if self.token_has_expired(response):
            # We got an 'Access token expired' response => refresh token
            self.obtain_token()
            # Re-dispatch the request that previously failed
            response = self.session.request(method, url, **kwargs)
        return response

    def token_has_expired(self, response):
        status = response.status_code
        content_type = response.headers['Content-Type']
        repJson = response.text
        if status == 401 and 'application/json' in content_type:
            repJson = response.text
            if 'Invalid JWT token' in repJson['description']:
                return True
        return False


    def obtain_token(self):
        # Obtain new token
        data = {'grant_type': 'client_credentials'}
        headers = {'Authorization': 'Basic ' + APPLICATION_ID}
        access_token_response = requests.post(TOKEN_URL, data=data, verify=False, allow_redirects=False, headers=headers)
        token = access_token_response.json()['access_token']
        # Update session with fresh token
        self.session.headers.update({'Authorization': 'Bearer %s' % token})

    def recuperer_cartes_climatique_temperature_24h(self,heure):
        #On récupère la date du jour, c'est à dire l'heure où a été générée le modèle AROME à 00h00
        #Pour récupérer les valeurs du 08
        datejour = datetime.datetime.today()
        #datejour = datetime.datetime.today()-datetime.timedelta(days=1)
        #On remet à 0 les heures, min ...
        datedujour = datejour.replace(hour=0, minute=0, second=0, microsecond=0,)
        date_du_modele = datedujour.isoformat()
        #On recupère la liste des stations climatique
        listes_stations = spark.read.format("org.apache.spark.sql.cassandra").options(table="stations_par_region", keyspace="consom").load() 
        listes_stations_GE = listes_stations.select('id_station','latitude','longitude').where(F.col('region')==F.lit('Grand Est'))
        listes_stations_GE.cache()
        # On définit une liste afin de récupérer toutes les cartes 24h dans la même journée : 
        date_heure_carte = datedujour+datetime.timedelta(hours=heure)
        date_heure_carte = date_heure_carte.isoformat()
        #On recupère la carte :
        response =  self.request('GET',"https://public-api.meteofrance.fr/public/arome/1.0/wcs/MF-NWP-HIGHRES-AROME-001-FRANCE-WCS/GetCoverage?service=WCS&version=2.0.1&coverageid=TEMPERATURE__SPECIFIC_HEIGHT_LEVEL_ABOVE_GROUND___"+date_du_modele+"Z&subset=height%282%29%26subset%3Dtime%28"+date_heure_carte+"Z%29&format=application%2Fwmo-grib", timeout=10,verify=False)
        if response.status_code == 200 :
            conversion_grib2(response,date_heure_carte,listes_stations_GE)
        else :
            print("erreur avec la carte du"+date_heure_carte)
        print("carte "+date_heure_carte+" faite")
        listes_stations_GE.unpersist()

def main():
    for heure in range(24):
        client = Client()
        client.session.headers.update({'Accept': 'application/json'})
        client.recuperer_cartes_climatique_temperature_24h(heure)


if __name__ == '__main__':
    main()

In [7]:
spark.stop()